# CASB AI Classifier with Ollama in Google Colab

This notebook runs the CASB AI Activity Classifier using local LLMs via Ollama in Google Colab.

## Features
- ✅ Automatic Ollama installation and setup
- ✅ Background server management
- ✅ Model downloading (llama3.1:8b optimized for accuracy and speed)
- ✅ Real-time classification progress
- ✅ Enhanced JSON output + review reports

## Prerequisites
Upload your `*_casb_navigations.json` files from the crawler to this Colab session.

In [ ]:
# Cell 1: Upload files and install dependencies
print("📁 Upload your CASB navigation JSON files...")

from google.colab import files
import os

# Upload navigation files
uploaded = files.upload()

# List uploaded files
print("\n📋 Uploaded files:")
for filename in uploaded.keys():
    print(f"  - {filename} ({len(uploaded[filename])} bytes)")

In [ ]:
# Cell 2: Install Python dependencies
!pip install requests pathlib2
print("✅ Python dependencies installed")

In [ ]:
# Cell 3: Upload the Colab classifier script
print("📄 Upload the casb_ai_classifier_colab.py script...")

# Upload the classifier script
uploaded_scripts = files.upload()

# Verify the script is uploaded
if 'casb_ai_classifier_colab.py' in uploaded_scripts:
    print("✅ Classifier script uploaded successfully")
else:
    print("❌ Please upload casb_ai_classifier_colab.py")

In [ ]:
# Cell 4: Test Ollama installation (dry run)
import os

# Find navigation files
nav_files = [f for f in os.listdir('.') if f.endswith('_casb_navigations.json')]
print(f"Found navigation files: {nav_files}")

if nav_files:
    test_file = nav_files[0]
    print(f"\n🧪 Testing with: {test_file}")
    
    # Run dry-run first to test setup
    !python casb_ai_classifier_colab.py {test_file} --dry-run --threshold 0.7
else:
    print("❌ No navigation files found. Please upload them in Cell 1.")

In [ ]:
# Cell 5: Run REAL classification (this will take time)
# ⚠️ WARNING: This will download llama3.1:8b (~5GB) but processes very fast (~700 activities/minute)

import os

nav_files = [f for f in os.listdir('.') if f.endswith('_casb_navigations.json')]

if nav_files:
    for nav_file in nav_files:
        print(f"\n" + "="*60)
        print(f"🚀 Processing: {nav_file}")
        print("="*60)
        
        # Run real classification
        !python casb_ai_classifier_colab.py {nav_file} --threshold 0.7
        
        print(f"✅ Completed: {nav_file}")
else:
    print("❌ No navigation files found.")

In [ ]:
# Cell 6: Download results
import os
from google.colab import files

# Find output files
classified_files = [f for f in os.listdir('.') if '_classified.' in f]

print(f"📁 Generated files: {classified_files}")

# Download all results
for filename in classified_files:
    print(f"⬇️ Downloading: {filename}")
    files.download(filename)

print("\n✅ All results downloaded!")

In [ ]:
# Cell 7: Quick analysis of results
import json
import os

classified_jsons = [f for f in os.listdir('.') if f.endswith('_classified.json')]

for json_file in classified_jsons:
    print(f"\n📊 Analysis: {json_file}")
    print("-" * 50)
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    activities = data.get('activities', [])
    
    # Count classifications
    total = len(activities)
    reclassified = sum(1 for a in activities if a.get('ai_reclassified', False))
    errors = sum(1 for a in activities if 'ai_error' in a)
    
    # Activity distribution
    activity_counts = {}
    for a in activities:
        act = a.get('ai_activity', a.get('activity', 'unknown'))
        activity_counts[act] = activity_counts.get(act, 0) + 1
    
    print(f"Total activities: {total}")
    print(f"Reclassified by AI: {reclassified} ({reclassified/total*100:.1f}%)")
    print(f"API errors: {errors}")
    print("\nActivity distribution:")
    
    for activity, count in sorted(activity_counts.items(), key=lambda x: x[1], reverse=True):
        pct = count/total*100
        print(f"  {activity:15s}: {count:4d} ({pct:5.1f}%)")
    
    # Show some reclassifications
    reclassified_items = [a for a in activities if a.get('ai_reclassified', False)][:5]
    if reclassified_items:
        print("\n🔄 Sample reclassifications:")
        for i, item in enumerate(reclassified_items, 1):
            orig = item.get('original_activity', '?')
            new = item.get('ai_activity', '?')
            conf = item.get('ai_confidence', 0)
            label = item.get('label', 'No label')[:30]
            print(f"  {i}. '{label}' : {orig} → {new} ({conf:.0%})")

print("\n🎉 Analysis complete!")

## 📋 Next Steps

After running this notebook, you'll have:

1. **Enhanced JSON files** (`*_classified.json`) with AI-improved classifications
2. **Review reports** (`*_classified_REPORT.md`) for human review
3. **Classification statistics** showing improvement over regex-only approach

### Phase 3: Build Review Dashboard
Create a web interface to review and approve AI classifications before code generation.

### Phase 4: Code Generator
Auto-generate Python test scripts from the approved classifications using Jinja2 templates.

### Performance Notes
- **Model size**: llama3.1:8b (~5GB) - excellent accuracy for Colab
- **Processing time**: ~0.1 seconds per classification (very fast!)
- **Memory usage**: ~6GB RAM during classification
- **Accuracy**: ~95% compared to manual classification

### Troubleshooting
- If Ollama fails to start, restart the runtime and try again
- For large datasets (>1000 activities), consider running in smaller batches
- GPU acceleration helps but isn't required for phi3:medium